In [ ]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget $PREFIX/download.py
!wget $PREFIX/embedder.py

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q1

In [10]:
q1 = "How does approximate nearest neighbor search work?"
v1 = model.encode(q1)
print(v1[0])

-0.020582044


Q2

In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
import numpy as np
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc['filename'] == target_filename)
content = target_doc['content']
v2 = model.encode(content)

cosine = np.dot(v1, v2)
print(f"{cosine:.2f}")

0.46


Q3

In [ ]:
import numpy as np
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["content"] for chunk in chunks]
X = model.encode(texts, batch_size=32, convert_to_numpy=True)

scores = [v1.dot(X[i]) for i in range(len(X))]
idx = np.argmax(scores)

print(f"Index: {idx}, Score: {scores[idx]:.3f}")
print(f"File: {chunks[idx]['filename']}")

Index: 94, Score: 0.562
File: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [ ]:
from tqdm.auto import tqdm

texts = [chunk['content'] for chunk in chunks]
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

scores = [v1.dot(X[i]) for i in range(len(X))]
idx = np.argmax(scores)

print(f"Index: {idx}, Score: {scores[idx]:.3f}")
print(f"Highest-scoring chunk file: {chunks[idx]['filename']}")

  0%|          | 0/6 [00:00<?, ?it/s]

Index: 94, Score: 0.562
Highest-scoring chunk file: 02-vector-search/lessons/07-sqlitesearch-vector.md


Q4

In [28]:
from minsearch import VectorSearch
import numpy as np

vindex = VectorSearch(keyword_fields=['content'])

X_array = np.array(X)
vindex.fit(X_array, chunks)

query1 = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query1)

results = vindex.search(query_vector, num_results=5)

print(results[0]['filename'])

04-evaluation/lessons/01-intro.md


In [34]:
from minsearch import VectorSearch
import numpy as np

X = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query4 = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query4, normalize_embeddings=True)

results = vindex.search(query_vector, num_results=5)

for i, res in enumerate(results):
    print(f"Rank {i+1}: {res['filename']}")

Rank 1: 04-evaluation/lessons/01-intro.md
Rank 2: 04-evaluation/lessons/05-search-metrics.md
Rank 3: 04-evaluation/lessons/05-search-metrics.md
Rank 4: 01-agentic-rag/lessons/05-search.md
Rank 5: 01-agentic-rag/lessons/05-search.md


Q5

In [35]:
from minsearch import Index, VectorSearch
import numpy as np

tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

vindex = VectorSearch(keyword_fields=['filename'])
X = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)
vindex.fit(X, chunks)

query_q5 = "How do I store vectors in PostgreSQL?"

text_results = tindex.search(query=query_q5, num_results=5)
text_files = [res['filename'] for res in text_results]

query_vector_q5 = model.encode(query_q5, normalize_embeddings=True)
vector_results = vindex.search(query_vector_q5, num_results=5)
vector_files = [res['filename'] for res in vector_results]

print("=== TOP 5 TEXT SEARCH RESULTS ===")
for i, f in enumerate(text_files):
    print(f"{i+1}. {f}")

print("\n=== TOP 5 VECTOR SEARCH RESULTS ===")
for i, f in enumerate(vector_files):
    print(f"{i+1}. {f}")

print("\n=== ANSWER CANDIDATES (In Vector but NOT in Text) ===")
difference = [f for f in vector_files if f not in text_files]
for f in difference:
    print(f)

=== TOP 5 TEXT SEARCH RESULTS ===
1. 02-vector-search/lessons/02-embeddings.md
2. 03-orchestration/lessons/05-rag.md
3. 02-vector-search/lessons/01-intro.md
4. 03-orchestration/lessons/05-rag.md
5. 02-vector-search/lessons/01-intro.md

=== TOP 5 VECTOR SEARCH RESULTS ===
1. 02-vector-search/lessons/08-pgvector.md
2. 02-vector-search/lessons/08-pgvector.md
3. 02-vector-search/lessons/08-pgvector.md
4. 02-vector-search/lessons/08-pgvector.md
5. 02-vector-search/lessons/08-pgvector.md

=== ANSWER CANDIDATES (In Vector but NOT in Text) ===
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


Q6

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query_q6 = "How do I give the model access to tools?"

query_vector_q6 = model.encode(query_q6, normalize_embeddings=True)
vector_results = vindex.search(query_vector_q6, num_results=5)

text_results = tindex.search(query=query_q6, num_results=5)

hybrid_results = rrf([vector_results, text_results])

print(f"The top-ranked file after RRF is: {hybrid_results[0]['filename']}")

The top-ranked file after RRF is: 01-agentic-rag/lessons/14-agentic-loop.md
